## ConversationBufferMemory

In [1]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

import warnings
warnings.filterwarnings('ignore')

In [2]:
# account for deprecation of LLM model
import datetime
# Get the current date
current_date = datetime.datetime.now().date()

# Define the date after which the model should be set to "gpt-3.5-turbo"
target_date = datetime.date(2024, 6, 12)

# Set the model variable based on the current date
if current_date > target_date:
    llm_model = "gpt-3.5-turbo"
else:
    llm_model = "gpt-3.5-turbo-0301"

In [ ]:
#!pip install langchain_community

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage


#Deprecated
#from langchain.chat_models import ChatOpenAI   
#from langchain.chains import ConversationChain
#from langchain.memory import ConversationBufferMemory



from langchain_openai import ChatOpenAI


llm = ChatOpenAI(
    model=llm_model,
    temperature=0.0,
    openai_api_key=os.environ.get("OPENAI_API_KEY")
)

messages = []

def chat(user_input: str):
    messages.append(HumanMessage(content=user_input))

    response = llm.invoke(messages)

    messages.append(AIMessage(content=response.content))

    return response.content


print(chat("Hi, my name is Andrew"))


Hello Andrew, nice to meet you! How can I assist you today?


In [20]:
print(chat("What is 1+1?"))

1+1 equals 2.


In [21]:
print(chat("What's my name?"))

Your name is Andrew.


In [24]:
print(messages)

[HumanMessage(content='Hi, my name is Andrew', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello Andrew, nice to meet you! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is 1+1?', additional_kwargs={}, response_metadata={}), AIMessage(content='1+1 equals 2.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="What's my name?", additional_kwargs={}, response_metadata={}), AIMessage(content='Your name is Andrew.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [25]:
for m in messages:
    print(f"{m.type}: {m.content}")


human: Hi, my name is Andrew
ai: Hello Andrew, nice to meet you! How can I assist you today?
human: What is 1+1?
ai: 1+1 equals 2.
human: What's my name?
ai: Your name is Andrew.


In [26]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import get_buffer_string

memory = InMemoryChatMessageHistory()

memory.add_user_message("Hi")
memory.add_ai_message("What's up")

print(get_buffer_string(memory.messages))   # equivalent to memory.buffer
print({"history": get_buffer_string(memory.messages)})  # equivalent to load_memory_variables({})

memory.add_user_message("Not much, just hanging")
memory.add_ai_message("Cool")

print(get_buffer_string(memory.messages))
print({"history": get_buffer_string(memory.messages)})


Human: Hi
AI: What's up
{'history': "Human: Hi\nAI: What's up"}
Human: Hi
AI: What's up
Human: Not much, just hanging
AI: Cool
{'history': "Human: Hi\nAI: What's up\nHuman: Not much, just hanging\nAI: Cool"}


## ConversationBufferWindowMemory

In [8]:
from langchain_openai import ChatOpenAI

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import trim_messages

In [13]:
llm_model = "gpt-3.5-turbo"

In [14]:
llm = ChatOpenAI(temperature=0.0, model=llm_model)

In [15]:
# Store for session histories (swap for a DB-backed store in production)
store = {}

In [16]:
def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [17]:
# Windowing: keep only the last k exchanges (mimics k=1 window)
trimmer = trim_messages(
    max_tokens=1,       # counts "messages" here via token_counter below
    strategy="last",
    token_counter=lambda msgs: len(msgs),  # counts messages, not tokens
    include_system=True,
)
# For k=1 (last 1 human+ai pair = 2 messages), set max_tokens=2

In [18]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

In [19]:
chain = prompt | llm

In [20]:
conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

In [22]:
config = {"configurable": {"session_id": "abc1"}}

In [23]:
conversation.invoke({"input": "Hi, my name is Andrew"}, config=config)
conversation.invoke({"input": "What is 1+1?"}, config=config)
conversation.invoke({"input": "What is my name?"}, config=config)

AIMessage(content='Your name is Andrew.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 5, 'prompt_tokens': 69, 'total_tokens': 74, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-ECb7KhMnuWiZjSKrz0uDpBtLGIA4G', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ffde8-d250-7140-aeeb-50dd98d29ba7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 69, 'output_tokens': 5, 'total_tokens': 74, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## TRIM MESSAGES Former ConversationTokenBufferMemory

In [ ]:
#!pip install tiktoken

In [27]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, trim_messages

In [28]:
llm = ChatOpenAI(temperature=0.0, model=llm_model)

In [29]:
# Build up a message list the same way memory.save_context did
messages = [
    HumanMessage(content="AI is what?!"),
    AIMessage(content="Amazing!"),
    HumanMessage(content="Backpropagation is what?"),
    AIMessage(content="Beautiful!"),
    HumanMessage(content="Chatbots are what?"),
    AIMessage(content="Charming!"),
]

In [30]:
trimmer = trim_messages(
    messages,
    max_tokens=50,
    strategy="last",
    token_counter=llm,   # uses the model's own tokenizer to count
    start_on="human",    # keeps pairs intact, starting on a human turn
)

In [31]:
trimmer  # equivalent to memory.load_memory_variables({})

[HumanMessage(content='AI is what?!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Amazing!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Backpropagation is what?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Beautiful!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Chatbots are what?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Charming!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

## Conversation Summary Buffer manually

In [32]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, trim_messages
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [33]:
llm = ChatOpenAI(temperature=0.0, model=llm_model)

In [34]:
schedule = "There is a meeting at 8am with your product team. \
You will need your powerpoint presentation prepared. \
9am-12pm have time to work on your LangChain \
project which will go quickly because Langchain is such a powerful tool. \
At Noon, lunch at the italian resturant with a customer who is driving \
from over an hour away to meet you to understand the latest in AI. \
Be sure to bring your laptop to show the latest LLM demo."

In [35]:
class SummaryBufferHistory(InMemoryChatMessageHistory):
    """Chat history that summarizes old messages once max_token_limit is exceeded."""
    llm: ChatOpenAI = None
    max_token_limit: int = 100
    running_summary: str = ""

    def add_message(self, message) -> None:
        super().add_message(message)
        self._condense()

    def _condense(self):
        # if we're under budget, do nothing
        token_count = self.llm.get_num_tokens_from_messages(self.messages)
        if token_count <= self.max_token_limit:
            return

        # split: keep trimming from the end backwards until under budget,
        # summarize everything that gets pushed out
        kept = trim_messages(
            self.messages,
            max_tokens=self.max_token_limit,
            strategy="last",
            token_counter=self.llm,
            start_on="human",
        )
        to_summarize = self.messages[: len(self.messages) - len(kept)]
        if not to_summarize:
            return

        summary_prompt = (
            f"Progressively summarize the conversation so far.\n\n"
            f"Current summary:\n{self.running_summary}\n\n"
            f"New lines to add:\n"
            + "\n".join(f"{m.type}: {m.content}" for m in to_summarize)
        )
        self.running_summary = self.llm.invoke(summary_prompt).content

        self.messages = [SystemMessage(content=f"Summary of earlier conversation: {self.running_summary}")] + kept

In [36]:
store = {}

In [37]:
def get_session_history(session_id: str) -> SummaryBufferHistory:
    if session_id not in store:
        store[session_id] = SummaryBufferHistory(llm=llm, max_token_limit=100)
    return store[session_id]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

In [38]:
chain = prompt | llm

In [39]:
conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

In [40]:
config = {"configurable": {"session_id": "abc1"}}

In [41]:
conversation.invoke({"input": "Hello"}, config=config)
conversation.invoke({"input": "Not much, just hanging"}, config=config)
conversation.invoke({"input": "What is on the schedule today?"}, config=config)

AIMessage(content="I'm here to assist you with any questions or tasks you have. Just let me know what you need help with, and I'll do my best to provide you with the information or support you're looking for.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 81, 'total_tokens': 124, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-ECbfH6PUPPTc1FX1NxWFZYNms5f53', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ffe08-f21b-74b1-93d6-0bcf9a37a4cf-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 81, 'output_tokens': 43, 'total_tokens': 124, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_to

In [42]:
# inspect memory state, equivalent to memory.load_memory_variables({})
get_session_history("abc1").messages

[SystemMessage(content='Summary of earlier conversation: The human initiates the conversation by saying "Hello." The AI responds politely and asks how it can assist the human today.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Not much, just hanging', additional_kwargs={}, response_metadata={}),
 AIMessage(content="That's nice! If you have any questions or need assistance with anything, feel free to ask. I'm here to help!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 40, 'total_tokens': 66, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-ECbfFC6QdBPsXEatbMzdAiu14noRJ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs':

In [43]:
# ask the follow-up, which needs the schedule context
conversation.invoke({"input": "What would be a good demo to show?"}, config=config)

AIMessage(content="That's a great question! The choice of a demo depends on the audience and the purpose of the demonstration. Here are a few ideas for demos that are generally well-received:\n\n1. Virtual Reality Experience: Showcasing a virtual reality experience can be engaging and immersive, especially if your audience is interested in technology or gaming.\n\n2. Product Demo: If you have a new product or service, a live demonstration can help showcase its features and benefits.\n\n3. Data Visualization: Presenting data in a visually appealing way can help make complex information more understandable. Tools like Tableau or Power BI can be used for this purpose.\n\n4. AI Chatbot Demo: Demonstrating an AI chatbot in action can be a fun and interactive way to showcase the capabilities of artificial intelligence.\n\n5. Interactive Website Demo: If you have a website or app, a live demo can help users understand its functionality and features.\n\nFeel free to provide more details about 

In [44]:
get_session_history("abc1").messages

[SystemMessage(content='Summary of earlier conversation: human: Thank you for the suggestions! I think a virtual reality experience would be a great demo to show. Can you provide more information on how to set that up?\nai: Of course! Setting up a virtual reality experience involves using VR headsets, controllers, and software to create an immersive environment. You can choose from a variety of VR platforms such as Oculus Rift, HTC Vive, or PlayStation VR, depending on your budget and technical requirements. Once you have the necessary equipment, you can download VR applications or games to showcase during the demo. I can help guide you through the setup process and provide tips on creating a memorable VR experience for your audience. Just let me know how I can assist you further!', additional_kwargs={}, response_metadata={})]